# Classifying with scikit-learn

In this notebook, you'll train a **classification** model. This is a machine learning model that predicts a *category* instead of a number.

<a target="_blank" href="https://codetto.app?github=simonguest/codercub/blob/main/labs/03/notebooks/sklearn_classification.ipynb">
  <img src="https://img.shields.io/badge/Open_in-Codetto-blue" alt="Open In Codetto"/>
</a>

## Part 1: Regression vs. Classification

So far you've seen **regression** in scikit-learn. This is predicting a *number* (like a test score).

**Classification** is different: you predict a *category*.

| Task | Type | Output |
|------|------|--------|
| Predict a student's test score | Regression | 84.5 |
| Predict whether a student passes | Classification | Pass / Fail |
| Predict tomorrow's temperature | Regression | 22°C |
| Predict whether it will rain | Classification | Rain / No Rain |

The good news: sklearn follows the **same three steps** for classification as it does for regression:
1. **Prepare** your features (`X`) and labels (`y`)
2. **Fit** — `model.fit(X, y)` — the model learns from your data
3. **Predict** — `model.predict(...)` — apply what it learned to new data

## Part 2: Our Dataset

We'll ask the question: *can a machine predict whether a student passes based on how much they studied AND how much they slept?*

This time we have **two features** — hours studied and hours of sleep — and a **label** of Pass (1) or Fail (0):

In [ ]:
import pandas as pd

data = {
  'hours_studied': [2, 3, 1, 7, 5, 6, 8, 9, 4, 7, 3, 6, 8, 2, 5, 4, 9, 1, 7, 6],
  'hours_slept':   [4, 5, 8, 4, 7, 8, 6, 7, 6, 7, 3, 5, 8, 7, 9, 4, 5, 5, 9, 6],
  'passed':        [0, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 1]
}

df = pd.DataFrame(data)
df['result'] = df['passed'].map({1: 'Pass', 0: 'Fail'})
df

## Features and Labels

We split our data into:
- **X** — the *features* (what we know): hours studied **and** hours slept
- **y** — the *label* (what we want to predict): pass or fail

With two features, our model can look at both study time and sleep together to make a smarter prediction:

In [ ]:
X = df[['hours_studied', 'hours_slept']]  # two features
y = df['passed']                           # 0 = Fail, 1 = Pass

print('X — our features:')
print(X.head())
print('\ny — our labels (0=Fail, 1=Pass):')
print(y.values)

## Part 3: Training a Model

We'll use **K-Nearest Neighbors (KNN)**. This is one of the most intuitive classifiers.

The idea is simple: *find the k students in your training data who are most similar to the new student, then vote on the outcome.*

With `n_neighbors=3`, the model looks at the **3 closest students** and takes a majority vote — if 2 out of 3 passed, it predicts Pass:

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

model = KNeighborsClassifier(n_neighbors=3)
model.fit(X, y)

# Check how well the model does on the training data
accuracy = model.score(X, y)
print('Model trained!')
print(f'Training accuracy: {accuracy * 100:.1f}%  (how often it gets the right answer)')

## Part 4: Making Predictions

Now we can ask the model to predict whether a *new* student will pass or fail.

Use the sliders to set the student's hours studied and hours slept, then see what the model predicts:

In [ ]:
HOURS_STUDIED = 5 #@param {type:"slider", min:1, max:10, step:1}
HOURS_SLEPT   = 7 #@param {type:"slider", min:1, max:10, step:1}

new_student = pd.DataFrame({'hours_studied': [HOURS_STUDIED], 'hours_slept': [HOURS_SLEPT]})
prediction = model.predict(new_student)[0]
result = 'Pass ✓' if prediction == 1 else 'Fail ✗'

print(f'Student studied for {HOURS_STUDIED} hours and slept for {HOURS_SLEPT} hours.')
print(f'Prediction: {result}')

## Part 5: Seeing the Data

With two features we can plot each student as a point on a chart — hours studied on one axis, hours slept on the other.

Green dots are students who **passed**. Red dots are students who **failed**. The star shows where your predicted student falls:

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(9, 6))

# Draw decision regions using a mesh
x_min, x_max = 0, 11
y_min, y_max = 0, 11
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
Z = model.predict(pd.DataFrame({'hours_studied': xx.ravel(), 'hours_slept': yy.ravel()}))
Z = Z.reshape(xx.shape)
ax.contourf(xx, yy, Z, alpha=0.15, levels=[-0.5, 0.5, 1.5], colors=['#e57373', '#81c784'])
ax.contour(xx, yy, Z, levels=[0.5], colors=['gray'], linewidths=1, linestyles='--')

# Plot the training data
passed = df[df['passed'] == 1]
failed = df[df['passed'] == 0]
ax.scatter(passed['hours_studied'], passed['hours_slept'], color='#2e7d32', s=80, zorder=5, label='Pass')
ax.scatter(failed['hours_studied'], failed['hours_slept'], color='#c62828', s=80, zorder=5, label='Fail')

# Mark the new student
star_color = '#2e7d32' if prediction == 1 else '#c62828'
ax.scatter(HOURS_STUDIED, HOURS_SLEPT, color=star_color, s=250, marker='*', zorder=6, label='Your student')

ax.set_xlabel('Hours Studied')
ax.set_ylabel('Hours Slept')
ax.set_title('Pass / Fail by Hours Studied and Hours Slept')
ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)
ax.legend()
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

Look at the decision boundary (the dashed line). Are there any students whose actual result surprises you given where they fall on the chart? What does this tell you about what it means when a machine learning model is wrong?

## Check Your Understanding